# 03 — Monitoring: Manifests, Drift, Error Clusters

Three monitoring artifacts that turn the pipeline from "runs once" into "runs reliably over time."

1. **Run manifest** — git sha + branch, file hashes for config/schema/inputs/outputs/gold, simulator seed/error_rate. Reproducibility envelope per run.
2. **Distribution drift** — PSI + Jensen-Shannon divergence on `category`, `priority`, `language`. Flags when the model's prediction distribution shifts away from a reference.
3. **Error clusters** — group rejected records by `(code, field_path)` with sample evidence and remediation hints.

Run from the repo root.

In [1]:
import json
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if (REPO_ROOT / "src" / "llm_etl_eval").is_dir():
    sys.path.insert(0, str(REPO_ROOT / "src"))
elif (REPO_ROOT.parent / "src" / "llm_etl_eval").is_dir():
    REPO_ROOT = REPO_ROOT.parent
    sys.path.insert(0, str(REPO_ROOT / "src"))

DATA_RAW = REPO_ROOT / "data" / "raw"
SCHEMA_PATH = REPO_ROOT / "data" / "expected" / "target_schema.json"
GOLD_PATH = REPO_ROOT / "data" / "gold" / "labels.jsonl"
BASELINE_CONFIG = REPO_ROOT / "configs" / "baseline.yaml"
IMPROVED_CONFIG = REPO_ROOT / "configs" / "improved.yaml"
WORK = REPO_ROOT / "notebooks" / "_work" / "03_monitoring"
WORK.mkdir(parents=True, exist_ok=True)

REPO_ROOT

WindowsPath('C:/Users/diogo/work_code/llm-etl-and-evaluation')

## Generate baseline + improved outputs

We need real `outputs_*.jsonl` and `rejected_*.jsonl` files to feed the monitoring modules. Same setup as notebook 02.

In [2]:
from llm_etl_eval.etl.run import run_pipeline
from llm_etl_eval.prompting.run import run_prompting
from llm_etl_eval.sim.llm_simulator import (
    SimulationConfig, load_inputs, simulate_records, write_outputs,
)
from llm_etl_eval.transform.run import run_transform


def run_config(label, config_path):
    base = WORK / label
    base.mkdir(parents=True, exist_ok=True)
    processed_dir = base / "processed"
    run_pipeline(DATA_RAW, processed_dir)
    inputs_path = processed_dir / "llm_inputs.jsonl"
    prompts_path = base / "prompts.jsonl"
    run_prompting(inputs_path, prompts_path, SCHEMA_PATH, config_path)
    outputs_path = base / "outputs.jsonl"
    write_outputs(
        simulate_records(load_inputs(inputs_path), SimulationConfig(error_rate=0.35, seed=7)),
        outputs_path,
    )
    final_dir = base / "final"
    run_transform(SCHEMA_PATH, outputs_path, final_dir, config_path)
    return {
        "inputs": inputs_path,
        "prompts": prompts_path,
        "outputs": outputs_path,
        "rejected": final_dir / "rejected.jsonl",
    }


baseline = run_config("baseline", BASELINE_CONFIG)
improved = run_config("improved", IMPROVED_CONFIG)
print("both runs complete")

both runs complete


## 1. Run manifest

A manifest captures everything needed to reproduce a run — git state, config/schema/inputs hashes, simulator settings. Stamped automatically next to each report by the demo scripts.

In [3]:
from llm_etl_eval.manifest.builder import ManifestInputs, build_manifest

manifest = build_manifest(
    ManifestInputs(
        config_path=IMPROVED_CONFIG,
        schema_path=SCHEMA_PATH,
        inputs_path=improved["inputs"],
        outputs_path=improved["outputs"],
        gold_path=GOLD_PATH,
        prompts_path=improved["prompts"],
        seed=7,
        error_rate=0.35,
        run_id="notebook-demo",
        label="improved",
    ),
    cwd=REPO_ROOT,
)

print(json.dumps(manifest, indent=2, sort_keys=True)[:1500])

{
  "config": {
    "exists": true,
    "path": "C:\\Users\\diogo\\work_code\\llm-etl-and-evaluation\\configs\\improved.yaml",
    "sha256": "4ae6fc1d137e0daba5bbc3fccd0983fafb6e53cd7f37fc178707931e85bc96e7",
    "size_bytes": 105
  },
  "created_at": "2026-05-02T21:24:27Z",
  "extra": {},
  "git": {
    "branch": "develop",
    "dirty": true,
    "sha": "59a16be931ee42ef05e21963a5ea005d81ae383e"
  },
  "gold": {
    "exists": true,
    "path": "C:\\Users\\diogo\\work_code\\llm-etl-and-evaluation\\data\\gold\\labels.jsonl",
    "records": 50,
    "sha256": "56ceef832a8bb0d292f41ba441784b045bfece59d117ab3bd7fe4511a2c530c9",
    "size_bytes": 7001
  },
  "inputs": {
    "exists": true,
    "path": "C:\\Users\\diogo\\work_code\\llm-etl-and-evaluation\\notebooks\\_work\\03_monitoring\\improved\\processed\\llm_inputs.jsonl",
    "records": 12,
    "sha256": "f981d01a0c025056c98d92094d7fdb796b3177b1bac6094844ef2c147b04c44d",
    "size_bytes": 4572
  },
  "label": "improved",
  "outputs": {
 

## 2. Distribution drift (PSI + JSD)

Compares categorical distributions between two runs across `category`, `priority`, `language`. PSI and Jensen-Shannon divergence (base 2, bounded [0, 1]) are reported per field with severity bands.

- PSI < 0.10 → stable
- 0.10–0.25 → moderate
- > 0.25 → significant

In [4]:
from llm_etl_eval.checks.distribution_drift import compute_drift


def load_jsonl(path):
    return [
        json.loads(line)
        for line in path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]


ref_records = load_jsonl(baseline["outputs"])
cand_records = load_jsonl(improved["outputs"])

report = compute_drift(
    ref_records, cand_records,
    fields=["category", "priority", "language"],
)

print(f"overall ok = {report.ok}\n")
print(f"{'field':<12} {'psi':>8} {'jsd':>8}  severity")
print("-" * 44)
for field in report.fields:
    print(
        f"{field.field_name:<12} "
        f"{field.psi:>8.4f} {field.jsd:>8.4f}  {field.severity}"
    )

overall ok = True

field             psi      jsd  severity
--------------------------------------------
category       0.0000   0.0000  stable
priority       0.0000   0.0000  stable
language       0.0000   0.0000  stable


### Inspect a per-field distribution

Each `FieldDrift` keeps the smoothed normalized distributions for both sides — good for explaining a breach to a reviewer.

In [5]:
category_drift = next(f for f in report.fields if f.field_name == "category")
print("reference (baseline) vs candidate (improved):\n")
print(f"{'value':<14} {'reference':>10} {'candidate':>10}")
print("-" * 38)
support = sorted(set(category_drift.reference) | set(category_drift.candidate))
for value in support:
    r = category_drift.reference.get(value, 0.0)
    c = category_drift.candidate.get(value, 0.0)
    print(f"{value:<14} {r:>10.3f} {c:>10.3f}")

reference (baseline) vs candidate (improved):

value           reference  candidate
--------------------------------------
account             0.083      0.083
billing             0.250      0.250
product             0.167      0.167
security            0.167      0.167
shipping            0.167      0.167
technical           0.167      0.167


## 3. Error clusters

Group rejected records by `(code, field_path)` so we see *where* failures live, not just *how many* there are. Each cluster carries sample messages, sample raw values, and a remediation hint mapped from the error code.

In [6]:
from llm_etl_eval.eval.error_clusters import cluster_rejections

rejections = load_jsonl(baseline["rejected"])
cluster_report = cluster_rejections(rejections)

print(f"rejected_records={cluster_report.totals['rejected_records']}  "
      f"total_errors={cluster_report.totals['total_errors']}  "
      f"distinct_clusters={cluster_report.totals['distinct_clusters']}")

rejected_records=4  total_errors=6  distinct_clusters=3


In [7]:
for cluster in cluster_report.clusters[:5]:
    field_label = cluster.field_path or "<root>"
    print(f"[{cluster.count}x / {cluster.affected_records} records] "
          f"{cluster.code} @ {field_label}")
    print(f"    remediation: {cluster.remediation}")
    if cluster.sample_messages:
        print(f"    sample: {cluster.sample_messages[0][:120]}")
    print()

[2x / 2 records] enum @ priority
    remediation: Value outside allowed set. Add an allowed-values reminder to the prompt.
    sample: 'critical' is not one of ['low', 'medium', 'high', 'urgent']

[2x / 2 records] invalid_type @ summary
    remediation: Wrong type for a field (or top-level non-object). Add a coercion rule or strengthen the schema reminder.
    sample: Expected string.

[2x / 2 records] missing_required @ summary
    remediation: Required field omitted. Reinforce the required-fields list in the prompt.
    sample: Missing required field.



## Putting it together

These three modules sit on top of the existing pipeline without touching any stage logic — they only consume the artifacts (`outputs.jsonl`, `rejected.jsonl`, `report.json`).

In CI ([.github/workflows/ci.yml](../.github/workflows/ci.yml)), the integration smoke test [tests/integration/test_demo_smoke.py](../tests/integration/test_demo_smoke.py) runs the entire chain and asserts every artifact's structure + metric floors, so silent regressions surface on every PR.